In [ ]:
# Cell 0 — Colab bootstrap (no-op when running locally)
import sys, os, subprocess, pathlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # If your repo is private, replace REPO with:
    #   f'https://{TOKEN}@github.com/sathira10/ds1-neural-emulation.git'
    REPO   = 'https://github.com/sathira10/ds1-neural-emulation.git'
    target = pathlib.Path('/content/ds1-neural-emulation')
    if not target.exists():
        subprocess.run(['git', 'clone', '--recursive', REPO, str(target)], check=True)
    os.chdir(target / 'model')
    print(f'Colab — working dir: {os.getcwd()}')
else:
    print(f'Local — working dir: {os.getcwd()}')

In [ ]:
# Cell 1 — Imports
import CoreAudioML.miscfuncs as miscfuncs
import CoreAudioML.training as training
import CoreAudioML.dataset as dataset
import CoreAudioML.networks as networks
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import json, os, time
from scipy.io.wavfile import write, read as wav_read
from IPython.display import Audio, display, Markdown, clear_output
from tqdm.auto import tqdm

In [ ]:
# Cell 1b — CoreAudioML GPU patches (no submodule edits needed)
import CoreAudioML.training as _caml
import CoreAudioML.networks as _caml_nets

# PreEmph.forward creates torch.zeros() on CPU; patch to respect the input device.
def _preemph_forward(self, output, target):
    dev = output.device
    output = torch.cat((torch.zeros(self.zPad, output.shape[1], 1, device=dev), output))
    target = torch.cat((torch.zeros(self.zPad, target.shape[1], 1, device=dev), target))
    output = self.conv_filter(output.permute(1, 2, 0))
    target = self.conv_filter(target.permute(1, 2, 0))
    if self.low_pass:
        output = self.lp_filter(output)
        target = self.lp_filter(target)
    return output.permute(2, 0, 1), target.permute(2, 0, 1)

# LossWrapper holds loss_factors as CPU scalars; move them on the fly.
def _losswrapper_forward(self, output, target):
    loss = 0
    for i, losses in enumerate(self.loss_functions):
        loss += torch.mul(losses(output, target), self.loss_factors[i].to(output.device))
    return loss

# SimpleRNN.forward: cuDNN rejects non-contiguous input or hidden state tensors.
def _simplernn_forward(self, x):
    x = x.contiguous()
    if self.hidden is not None:
        if isinstance(self.hidden, tuple):
            self.hidden = tuple(h.contiguous() for h in self.hidden)
        else:
            self.hidden = self.hidden.contiguous()
    if self.skip:
        res = x[:, :, 0:self.skip]
        x, self.hidden = self.rec(x, self.hidden)
        return self.lin(x) + res
    else:
        x, self.hidden = self.rec(x, self.hidden)
        return self.lin(x)

_caml.PreEmph.forward = _preemph_forward
_caml.LossWrapper.forward = _losswrapper_forward
_caml_nets.SimpleRNN.forward = _simplernn_forward
print('CoreAudioML GPU patches applied.')

In [ ]:
# Cell 2 — Configuration
CONFIGS = [
    {
        'name': 'LSTM',
        'unit_type': 'LSTM',
        'hidden_size': 16,
        'num_blocks': 1,
        'skip_con': 1,
    },
    {
        'name': 'GRU',
        'unit_type': 'GRU',
        'hidden_size': 32,
        'num_blocks': 1,
        'skip_con': 1,
    },
]

DEVICES = ['ds1']

TRAIN_PARAMS = {
    'seed': 0,
    'epochs': 500,
    'learn_rate': 0.005,
    'batch_size': 50,
    'init_len': 200,
    'up_fr': 1000,
    'val_chunk': 100000,
    'test_chunk': 100000,
    'validation_f': 2,
    'validation_p': 25,
    'segment_length': 22050,
    'loss_fcns': {'ESRPre': 0.75, 'DC': 0.25},
    'pre_filt': [-0.85, 1],
    'input_size': 1,
    'output_size': 1,
    'data_location': './Data',
    'save_location': './Results',
}

FORCE_RETRAIN = False

In [ ]:
# Cell 3 — Run key helper
def run_key(device, config_name):
    return f"{device}_{config_name}"

In [ ]:
# Cell 4 — Data loading (once, shared across all configs)
datasets = {}

for device in DEVICES:
    ds = dataset.DataSet(data_dir=TRAIN_PARAMS['data_location'])

    ds.create_subset('train', frame_len=TRAIN_PARAMS['segment_length'])
    ds.load_file(os.path.join('train', device), 'train')

    ds.create_subset('val')
    ds.load_file(os.path.join('val', device), 'val')

    ds.create_subset('test')
    ds.load_file(os.path.join('test', device), 'test')

    datasets[device] = ds

    fs = ds.subsets['train'].fs
    train_in  = ds.subsets['train'].data['input'][0]
    train_tgt = ds.subsets['train'].data['target'][0]
    val_in    = ds.subsets['val'].data['input'][0]
    test_in   = ds.subsets['test'].data['input'][0]

    print(f"[{device}] sample rate: {fs} Hz")
    print(f"[{device}] train input  shape: {list(train_in.shape)}  (frame_len, n_segs, channels)")
    print(f"[{device}] train target shape: {list(train_tgt.shape)}")
    print(f"[{device}] val   input  shape: {list(val_in.shape)}")
    print(f"[{device}] test  input  shape: {list(test_in.shape)}")

    # Stack enough consecutive segments to fill ~2 s for the preview
    segs_for_2s = max(1, int(np.ceil(2 * fs / TRAIN_PARAMS['segment_length'])))
    n_preview_segs = min(segs_for_2s, train_in.shape[1])
    # train_in shape: (frame_len, n_segs, 1) — permute then flatten to (n_segs * frame_len,)
    inp_preview = train_in[:, :n_preview_segs, 0].permute(1, 0).reshape(-1).numpy()
    tgt_preview = train_tgt[:, :n_preview_segs, 0].permute(1, 0).reshape(-1).numpy()
    preview_samples = len(inp_preview)
    t = np.arange(preview_samples) / fs

    fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
    axes[0].plot(t, inp_preview, linewidth=0.5)
    axes[0].set_ylabel('Input')
    axes[0].set_title(f'{device} — training waveform preview (first {preview_samples / fs:.2f} s)')
    axes[1].plot(t, tgt_preview, linewidth=0.5, color='C1')
    axes[1].set_ylabel('Target')
    axes[1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 5 — Persistent stats helpers
STATS_DIR = './notebook_stats'
os.makedirs(STATS_DIR, exist_ok=True)

def stats_path(device, config_name):
    return os.path.join(STATS_DIR, f"{run_key(device, config_name)}_stats.json")

def save_stats(device, config_name, stats_dict):
    with open(stats_path(device, config_name), 'w') as f:
        json.dump(stats_dict, f, indent=2)

def load_stats(device, config_name):
    p = stats_path(device, config_name)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return None

In [ ]:
# Cell 6 — Model initialisation
def init_model(save_path, cfg):
    if miscfuncs.file_check('model.json', save_path):
        print(f'  existing model file found in {save_path}, loading network')
        model_data = miscfuncs.json_load('model', save_path)
        try:
            assert model_data['model_data']['unit_type'] == cfg['unit_type']
            assert model_data['model_data']['input_size'] == TRAIN_PARAMS['input_size']
            assert model_data['model_data']['hidden_size'] == cfg['hidden_size']
            assert model_data['model_data']['output_size'] == TRAIN_PARAMS['output_size']
        except AssertionError:
            print('  model.json architecture mismatch — creating fresh network')
            network = networks.SimpleRNN(
                input_size=TRAIN_PARAMS['input_size'],
                unit_type=cfg['unit_type'],
                hidden_size=cfg['hidden_size'],
                output_size=TRAIN_PARAMS['output_size'],
                skip=cfg['skip_con'],
            )
            network.save_state = False
            network.save_model('model', save_path)
            return network
        network = networks.load_model(model_data)
    else:
        print(f'  no saved model found in {save_path}, creating new network')
        network = networks.SimpleRNN(
            input_size=TRAIN_PARAMS['input_size'],
            unit_type=cfg['unit_type'],
            hidden_size=cfg['hidden_size'],
            output_size=TRAIN_PARAMS['output_size'],
            skip=cfg['skip_con'],
        )
        network.save_state = False
        network.save_model('model', save_path)
    return network

In [ ]:
# Cell 7 — Training loop

# --- Device detection ---
if torch.cuda.is_available():
    accel = 'cuda'
# elif torch.backends.mps.is_available():
#     accel = 'mps'
else:
    accel = 'cpu'
print(f'Accelerator: {accel}')


torch.manual_seed(TRAIN_PARAMS['seed'])
np.random.seed(TRAIN_PARAMS['seed'])

PLOT_EVERY = 5  # redraw the live dashboard every N epochs


def _draw_live(stats, key, total_epochs):
    """Live-updating training dashboard (clears and redraws in place)."""
    ep        = stats['total_epochs_run']
    train_now = stats['train_losses'][-1] if stats['train_losses'] else float('nan')
    lr_now    = stats['learning_rates'][-1] if stats['learning_rates'] else float('nan')
    best_val  = stats['best_val_loss']
    best_ep   = stats['best_epoch']
    best_str  = f'{best_val:.5f} @ ep {best_ep}' if best_ep else '–'

    clear_output(wait=True)

    fig = plt.figure(figsize=(14, 4.2))
    fig.suptitle(
        f'{key}   epoch {ep} / {total_epochs}   '
        f'train {train_now:.5f}   best val {best_str}   lr {lr_now:.2e}',
        fontsize=10, color='#333', y=1.01,
    )

    gs = fig.add_gridspec(1, 3, wspace=0.35)
    ax_train = fig.add_subplot(gs[0, 0])
    ax_val   = fig.add_subplot(gs[0, 1])
    ax_lr    = fig.add_subplot(gs[0, 2])

    # --- training loss ---
    x = list(range(1, len(stats['train_losses']) + 1))
    ax_train.plot(x, stats['train_losses'], color='#2196F3', lw=1.2, alpha=0.9)
    if best_ep:
        ax_train.axvline(best_ep, color='#4CAF50', lw=1.2, ls='--', alpha=0.8)
    ax_train.set_title('Train loss', fontsize=10)
    ax_train.set_xlabel('Epoch', fontsize=9)
    ax_train.grid(True, alpha=0.2, linewidth=0.6)
    ax_train.tick_params(labelsize=8)

    # --- validation loss ---
    if stats['val_losses']:
        val_ep, val_l = zip(*stats['val_losses'])
        ax_val.plot(list(val_ep), list(val_l), color='#FF5722', lw=1.4,
                    marker='o', markersize=2.5, markevery=max(1, len(val_ep)//30))
        if best_ep:
            ax_val.axvline(best_ep, color='#4CAF50', lw=1.2, ls='--', alpha=0.8,
                           label=f'best @ {best_ep}')
            ax_val.legend(fontsize=8, loc='upper right', framealpha=0.6)
    ax_val.set_title('Val loss', fontsize=10)
    ax_val.set_xlabel('Epoch', fontsize=9)
    ax_val.grid(True, alpha=0.2, linewidth=0.6)
    ax_val.tick_params(labelsize=8)

    # --- learning rate ---
    ax_lr.plot(range(1, len(stats['learning_rates']) + 1), stats['learning_rates'],
               color='#9C27B0', lw=1.2)
    ax_lr.set_title('Learning rate', fontsize=10)
    ax_lr.set_xlabel('Epoch', fontsize=9)
    ax_lr.grid(True, alpha=0.2, linewidth=0.6)
    ax_lr.tick_params(labelsize=8)
    ax_lr.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2e}'))

    plt.show()


def _move_loss_fn_to_device(loss_fn, device):
    """Move PreEmph (held in ESRPre lambda closure) to device."""
    if device == 'cpu':
        return
    for fn in loss_fn.loss_functions:
        if callable(fn) and not isinstance(fn, torch.nn.Module):
            for cell in (getattr(fn, '__closure__', None) or []):
                try:
                    obj = cell.cell_contents
                    if isinstance(obj, torch.nn.Module):
                        obj.to(device)
                except (ValueError, AttributeError):
                    pass


def _write_audio(path, fs, tensor):
    """Write tensor to wav, normalising peak if it exceeds 1.0."""
    arr = tensor.cpu().numpy()[:, 0, 0]
    peak = np.abs(arr).max()
    if peak > 1.0:
        arr = arr / peak
    write(path, fs, arr)


In [ ]:
# Cell 8 — Training

# --- Training ---
all_stats = {}

for device in DEVICES:
    ds = datasets[device]

    train_in_  = ds.subsets['train'].data['input'][0].to(accel)
    train_tgt_ = ds.subsets['train'].data['target'][0].to(accel)
    val_in_    = ds.subsets['val'].data['input'][0].to(accel)
    val_tgt_   = ds.subsets['val'].data['target'][0].to(accel)
    test_in_   = ds.subsets['test'].data['input'][0].to(accel)
    test_tgt_  = ds.subsets['test'].data['target'][0].to(accel)

    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        save_path = os.path.join(TRAIN_PARAMS['save_location'], f"{device}-{cfg['name']}")
        os.makedirs(save_path, exist_ok=True)

        existing = load_stats(device, cfg['name'])
        if existing and not FORCE_RETRAIN:
            print(f"[{key}] Stats found on disk — skipping. (Set FORCE_RETRAIN=True to override.)")
            all_stats[key] = existing
            continue

        print(f"\n[{key}] Starting training  ({accel})")
        network = init_model(save_path, cfg).to(accel)
        network.save_state = True

        optimiser = optim.Adam(network.parameters(), lr=TRAIN_PARAMS['learn_rate'], weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, 'min', factor=0.5, patience=5)
        loss_fn   = training.LossWrapper(TRAIN_PARAMS['loss_fcns'], TRAIN_PARAMS['pre_filt'])
        _move_loss_fn_to_device(loss_fn, accel)
        esr_loss  = training.ESRLoss()

        stats = {
            'config_name': cfg['name'],
            'device': device,
            'train_losses': [],
            'val_losses': [],
            'learning_rates': [],
            'best_val_loss': float('inf'),
            'best_epoch': 0,
            'test_loss_final': None,
            'test_loss_esr_final': None,
            'test_loss_best': None,
            'test_loss_esr_best': None,
            'total_epochs_run': 0,
            'stopped_early': False,
            'training_time_seconds': 0.0,
        }

        patience_counter = 0
        total_epochs = TRAIN_PARAMS['epochs']

        for epoch in tqdm(range(1, total_epochs + 1), desc=key, leave=False):
            ep_start = time.time()

            epoch_loss = network.train_epoch(
                train_in_, train_tgt_,
                loss_fn, optimiser,
                TRAIN_PARAMS['batch_size'],
                TRAIN_PARAMS['init_len'],
                TRAIN_PARAMS['up_fr'],
            )

            current_lr = optimiser.param_groups[0]['lr']
            stats['train_losses'].append(epoch_loss.item())
            stats['learning_rates'].append(current_lr)
            stats['total_epochs_run'] = epoch
            stats['training_time_seconds'] += time.time() - ep_start

            val_loss_val = None
            if epoch % TRAIN_PARAMS['validation_f'] == 0:
                val_output, val_loss = network.process_data(
                    val_in_, val_tgt_, loss_fn, TRAIN_PARAMS['val_chunk'],
                )
                val_loss_val = val_loss.item()
                scheduler.step(val_loss)
                stats['val_losses'].append([epoch, val_loss_val])

                if val_loss_val < stats['best_val_loss']:
                    stats['best_val_loss'] = val_loss_val
                    stats['best_epoch'] = epoch
                    patience_counter = 0
                    network.save_model('model_best', save_path)
                    _write_audio(os.path.join(save_path, 'best_val_out.wav'), ds.subsets['val'].fs, val_output)
                else:
                    patience_counter += 1

            network.save_model('model', save_path)
            save_stats(device, cfg['name'], stats)

            if epoch % PLOT_EVERY == 0:
                _draw_live(stats, key, total_epochs)

            if TRAIN_PARAMS['validation_p'] and patience_counter > TRAIN_PARAMS['validation_p']:
                stats['stopped_early'] = True
                print(f"[{key}] Early stopping at epoch {epoch}")
                break

        # Final plot for this run
        _draw_live(stats, key, total_epochs)

        # Test evaluation — final-epoch model
        test_output, test_loss = network.process_data(
            test_in_, test_tgt_, loss_fn, TRAIN_PARAMS['test_chunk'],
        )
        test_esr = esr_loss(test_output, test_tgt_)
        _write_audio(os.path.join(save_path, 'test_out_final.wav'), ds.subsets['test'].fs, test_output)
        stats['test_loss_final']     = test_loss.item()
        stats['test_loss_esr_final'] = test_esr.item()

        # Test evaluation — best-validation model
        best_val_net = miscfuncs.json_load('model_best', save_path)
        best_network = networks.load_model(best_val_net).to(accel)
        test_output_best, test_loss_best = best_network.process_data(
            test_in_, test_tgt_, loss_fn, TRAIN_PARAMS['test_chunk'],
        )
        test_esr_best = esr_loss(test_output_best, test_tgt_)
        _write_audio(os.path.join(save_path, 'test_out_bestv.wav'), ds.subsets['test'].fs, test_output_best)
        stats['test_loss_best']     = test_loss_best.item()
        stats['test_loss_esr_best'] = test_esr_best.item()

        save_stats(device, cfg['name'], stats)
        all_stats[key] = stats
        print(f"[{key}] Done.  Test ESR — final: {stats['test_loss_esr_final']:.5f}  best-val: {stats['test_loss_esr_best']:.5f}")

In [ ]:
# Cell 9 — Load all stats from disk (run this independently after kernel restart)
# Requires: cells 1–3 and 5 to have been run first.
all_stats = {}
for device in DEVICES:
    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        stats = load_stats(device, cfg['name'])
        if stats:
            all_stats[key] = stats
            print(f"[{key}] loaded — {stats['total_epochs_run']} epochs, best val={stats['best_val_loss']:.6f}")
        else:
            print(f"[{key}] No stats found on disk.")

In [ ]:
# Cell 10 — Loss curves
for device in DEVICES:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'{device} — loss curves')

    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        if key not in all_stats:
            continue
        s = all_stats[key]

        epochs = range(1, len(s['train_losses']) + 1)
        axes[0].plot(list(epochs), s['train_losses'], label=cfg['name'])
        if s['best_epoch']:
            axes[0].axvline(s['best_epoch'], linestyle='--', alpha=0.5)

        if s['val_losses']:
            val_ep, val_l = zip(*s['val_losses'])
            axes[1].plot(val_ep, val_l, label=cfg['name'])
            if s['best_epoch']:
                axes[1].axvline(s['best_epoch'], linestyle='--', alpha=0.5)

    axes[0].set_title('Training loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].set_title('Validation loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 11 — Learning rate decay
for device in DEVICES:
    fig, ax = plt.subplots(figsize=(10, 3))
    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        if key not in all_stats:
            continue
        s = all_stats[key]
        ax.plot(range(1, len(s['learning_rates']) + 1), s['learning_rates'], label=cfg['name'])
    ax.set_title(f'{device} — learning rate over epochs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning rate')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 12 — Per-epoch metrics table
rows = []
for key, s in all_stats.items():
    val_by_epoch = {ep: loss for ep, loss in s['val_losses']}
    for i, (tl, lr) in enumerate(zip(s['train_losses'], s['learning_rates'])):
        ep = i + 1
        rows.append({
            'run_key': key,
            'epoch': ep,
            'train_loss': tl,
            'val_loss': val_by_epoch.get(ep, float('nan')),
            'lr': lr,
        })

df_epochs = pd.DataFrame(rows)

# Filter helper — edit device_filter or run_filter as needed
device_filter = DEVICES[0]
subset = df_epochs[df_epochs['run_key'].str.startswith(device_filter)]
display(subset.style.format({'train_loss': '{:.6f}', 'val_loss': '{:.6f}', 'lr': '{:.8f}'}).hide(axis='index'))

In [ ]:
# Cell 13 — Final metrics comparison table
summary_rows = []
for device in DEVICES:
    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        if key not in all_stats:
            continue
        s = all_stats[key]
        summary_rows.append({
            'Run': key,
            'Best Val Loss': s['best_val_loss'],
            'Best Epoch': s['best_epoch'],
            'Stopped Early': s['stopped_early'],
            'Test Loss (final)': s['test_loss_final'],
            'Test ESR (final)': s['test_loss_esr_final'],
            'Test Loss (best val)': s['test_loss_best'],
            'Test ESR (best val)': s['test_loss_esr_best'],
            'Training Time (s)': round(s['training_time_seconds'], 1),
        })

df_summary = pd.DataFrame(summary_rows)
fmt = {c: '{:.6f}' for c in ['Best Val Loss', 'Test Loss (final)', 'Test ESR (final)', 'Test Loss (best val)', 'Test ESR (best val)']}
display(df_summary.style.format(fmt).hide(axis='index'))

In [ ]:
# Cell 14 — Audio playback comparison
for device in DEVICES:
    target_path = os.path.join(TRAIN_PARAMS['data_location'], 'test', f'{device}-target.wav')
    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        save_path = os.path.join(TRAIN_PARAMS['save_location'], f"{device}-{cfg['name']}")
        display(Markdown(f'### {key}'))

        for label, path in [
            ('Ground truth (target)', target_path),
            ('Model output — final epoch', os.path.join(save_path, 'test_out_final.wav')),
            ('Model output — best val', os.path.join(save_path, 'test_out_bestv.wav')),
        ]:
            if os.path.exists(path):
                display(Markdown(f'**{label}**'))
                display(Audio(filename=path))
            else:
                display(Markdown(f'_{label}: file not found at `{path}`_'))

In [ ]:
# Cell 15 — Waveform and spectrogram comparison
def load_wav_mono(path):
    fs, data = wav_read(path)
    if data.dtype == 'int16':
        data = data.astype(np.float32) / 32768.0
    if data.ndim > 1:
        data = data[:, 0]
    return fs, data

PREVIEW_S = 2  # seconds of audio to show in waveform

for device in DEVICES:
    target_path = os.path.join(TRAIN_PARAMS['data_location'], 'test', f'{device}-target.wav')
    if not os.path.exists(target_path):
        print(f'Target not found: {target_path}')
        continue
    fs_tgt, tgt_audio = load_wav_mono(target_path)

    for cfg in CONFIGS:
        key = run_key(device, cfg['name'])
        save_path = os.path.join(TRAIN_PARAMS['save_location'], f"{device}-{cfg['name']}")
        out_path = os.path.join(save_path, 'test_out_bestv.wav')

        if not os.path.exists(out_path):
            print(f'[{key}] Output wav not found, skipping plot.')
            continue

        fs_out, out_audio = load_wav_mono(out_path)
        preview = PREVIEW_S * fs_tgt
        t = np.arange(preview) / fs_tgt

        fig, axes = plt.subplots(2, 2, figsize=(14, 7))
        fig.suptitle(f'{key} — waveform and spectrogram (best val model)')

        # Waveforms
        axes[0, 0].plot(t, tgt_audio[:preview], linewidth=0.5)
        axes[0, 0].set_title('Target — waveform')
        axes[0, 0].set_xlabel('Time (s)')

        axes[0, 1].plot(t, out_audio[:preview], linewidth=0.5, color='C1')
        axes[0, 1].set_title('Model output — waveform')
        axes[0, 1].set_xlabel('Time (s)')

        # Spectrograms
        axes[1, 0].specgram(tgt_audio, NFFT=1024, Fs=fs_tgt, noverlap=512)
        axes[1, 0].set_title('Target — spectrogram')
        axes[1, 0].set_xlabel('Time (s)')
        axes[1, 0].set_ylabel('Frequency (Hz)')

        axes[1, 1].specgram(out_audio, NFFT=1024, Fs=fs_out, noverlap=512)
        axes[1, 1].set_title('Model output — spectrogram')
        axes[1, 1].set_xlabel('Time (s)')
        axes[1, 1].set_ylabel('Frequency (Hz)')

        plt.tight_layout()
        plt.show()